# Baseline 6 — Aouarib et al. (2025): GA + XGBoost
**Reference:** Aouarib, Henouda, Laallam, *J. Inf. Sys. Eng. Mgmt.* 10(57s):1081–1092, 2025.---
### What this reproduces
Genetic algorithm jointly optimizes feature mask and XGBoost hyperparameters.We compare GA-XGB to plain XGB on each window, both labels.
### V2 setup notes (this re-run)
- Reads from **`dbo.tbl_graphgrued_weekly_active_base_balanced`** (V4 cohort + balanced label).
- Uses **TRUE weekly columns** `*_week1..*_week4` for windowed views (no linear scaling).
- Targets: `y_5class` (A=0..E=4) and `y_binary` (is_completed) — both already in V4.
- Stratified 8:1:1 split (seed=42), median-impute + standardize fit on train only.
### Model recipe- Pop=12, gens=5, tournament selection, single-point crossover, bit-flip mutation.
- Chromosome: feature mask + n_estimators / max_depth / lr / subsample / colsample.
- Fitness: validation Macro-F1.

In [3]:
# ============================================================
# 0) IMPORTS & CONFIG
# ============================================================
import os, time, json, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing  import StandardScaler, LabelEncoder
from sklearn.impute         import SimpleImputer
from sklearn.metrics        import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, classification_report, confusion_matrix
)

SEED = 42
np.random.seed(SEED)

MODEL_BASE_TABLE = "dbo.tbl_graphgrued_weekly_active_base_ordered5"
WINDOWS = [1, 2, 4]
LABEL_ORDER_5 = ["A", "B", "C", "D", "E"]

print("Imports OK — reading from", MODEL_BASE_TABLE)


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 42, Finished, Available, Finished, False)

Imports OK — reading from dbo.tbl_graphgrued_weekly_active_base_ordered5


In [4]:
# ============================================================
# 1) LOAD MODEL BASE TABLE (V4 balanced)
# ============================================================
SAMPLE_FRACTION = 1.0
MAX_ROWS = 300_000

sdf = spark.table(MODEL_BASE_TABLE)
total_rows = sdf.count()
print(f"V4 cohort rows: {total_rows:,}")

if SAMPLE_FRACTION < 1.0:
    sdf = sdf.sample(withReplacement=False, fraction=SAMPLE_FRACTION, seed=SEED)

pdf = sdf.toPandas()
if len(pdf) > MAX_ROWS:
    g = pdf.groupby("risk_label_5_idx", group_keys=False)
    pdf = g.apply(lambda d: d.sample(
        n=max(1, int(len(d) * MAX_ROWS / len(pdf))),
        random_state=SEED)).reset_index(drop=True)

print(f"Working dataframe shape: {pdf.shape}")
print("\n5-class distribution after sampling:")
print(pdf["risk_label_5_idx"]
      .map({0:"A",1:"B",2:"C",3:"D",4:"E"})
      .value_counts(normalize=True).round(4)
      .reindex(LABEL_ORDER_5))


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 43, Finished, Available, Finished, False)

V4 cohort rows: 100,409
Working dataframe shape: (100409, 71)

5-class distribution after sampling:
risk_label_5_idx
A    0.0222
B    0.0812
C    0.0769
D    0.6411
E    0.1785
Name: proportion, dtype: float64


In [5]:
# ============================================================
# 2) LABELS — already in V4; just alias
# ============================================================
pdf["y_5class"] = pdf["risk_label_5_idx"].astype(int)
pdf["y_binary"] = pdf["is_completed"].fillna(0).astype(int)

print("y_5class distribution:")
counts5 = pdf["y_5class"].value_counts().sort_index()
print(counts5)
min_share = counts5.min() / counts5.sum()
if min_share < 0.005:
    raise RuntimeError(
        f"Class collapse: smallest class is {min_share:.2%} < 0.5%. "
        f"Re-run 01_v4_relabel_balanced.ipynb with adjusted thresholds.")

print("\ny_binary distribution:")
print(pdf["y_binary"].value_counts(normalize=True).round(4))


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 45, Finished, Available, Finished, False)

y_5class distribution:
y_5class
0     2231
1     8156
2     7721
3    64375
4    17926
Name: count, dtype: int64

y_binary distribution:
y_binary
0    0.6692
1    0.3308
Name: proportion, dtype: float64


In [6]:
# ============================================================
# 3) FEATURE GROUPS — TRUE weekly columns + static
# ============================================================
def _by_suffix(suffix):
    return sorted([c for c in pdf.columns if c.endswith(suffix)])

WEEK_COLS = {
    1: _by_suffix("_week1"),
    2: _by_suffix("_week1") + _by_suffix("_week2"),
    3: _by_suffix("_week1") + _by_suffix("_week2") + _by_suffix("_week3"),
    4: _by_suffix("_week1") + _by_suffix("_week2") + _by_suffix("_week3") + _by_suffix("_week4"),
}

EXCLUDE = {
    "user_id", "course_id", "enroll_time",
    "is_completed", "completion_rate",
    "risk_label_5_name", "risk_label_5_idx",
    "label_D_flag", "label_E_flag",
    "y_binary", "y_5class",
    "total_4w_interactions", "post_w1_interactions", "perf_composite",
    "course_norm_score",
    "problems_passed", "course_raw_score_sum", "total_problems",
    "risk_label_5_surrogate",
}
ALL_WEEKLY = set().union(*WEEK_COLS.values())

STATIC_COLS = [
    c for c in pdf.columns
    if c not in EXCLUDE
    and c not in ALL_WEEKLY
    and pdf[c].dtype.kind in "biufc"
]

print(f"Static feature count: {len(STATIC_COLS)}")
print("Static cols:", STATIC_COLS[:20], "..." if len(STATIC_COLS) > 20 else "")
print()
for W, cols in WEEK_COLS.items():
    print(f"W={W}: {len(cols)} weekly columns up to and including week {W}")


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 46, Finished, Available, Finished, False)

Static feature count: 16
Static cols: ['year_of_birth', 'num_courses_enrolled', 'num_fields', 'num_resources', 'num_active_enrollments', 'course_total_videos', 'course_max_chapter', 'video_count_total_4w', 'video_watch_time_total_4w', 'chapter_coverage_ratio_raw', 'chapter_coverage_ratio', 'avg_chapter_progress_4w', 'chapter_progress_ratio_raw', 'chapter_progress_ratio', 'activity_signal_4w', 'is_active_4w'] 

W=1: 9 weekly columns up to and including week 1
W=2: 18 weekly columns up to and including week 2
W=3: 27 weekly columns up to and including week 3
W=4: 36 weekly columns up to and including week 4


In [7]:
# ============================================================
# 4) BUILD WINDOW VIEWS — leakage-safe (only weeks 1..W visible)
# ============================================================
def make_window_view(W):
    cols = STATIC_COLS + WEEK_COLS[W]
    cols = [c for c in cols if c in pdf.columns]
    Xdf  = pdf[cols].copy()
    for c in cols:
        Xdf[c] = pd.to_numeric(Xdf[c], errors="coerce").astype(float)
    return Xdf, cols

window_views = {}
for W in WINDOWS:
    Xdf, cols = make_window_view(W)
    window_views[W] = {"X": Xdf, "cols": cols}
    nan_pct = Xdf.isna().mean().mean()
    print(f"W={W}: shape={Xdf.shape}, mean NaN ratio={nan_pct:.3%}")


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 47, Finished, Available, Finished, False)

W=1: shape=(100409, 25), mean NaN ratio=27.326%
W=2: shape=(100409, 34), mean NaN ratio=43.076%
W=4: shape=(100409, 52), mean NaN ratio=59.910%


In [8]:
# ============================================================
# 5) STRATIFIED 8:1:1 SPLIT (per label)
# ============================================================
def split_idx(y, test_size=0.1, val_size=0.1, seed=SEED):
    idx = np.arange(len(y))
    idx_tv, idx_te = train_test_split(
        idx, test_size=test_size, stratify=y, random_state=seed)
    val_rel = val_size / (1.0 - test_size)
    idx_tr, idx_va = train_test_split(
        idx_tv, test_size=val_rel, stratify=y[idx_tv], random_state=seed)
    return idx_tr, idx_va, idx_te

splits = {}
for label in ["y_binary", "y_5class"]:
    y = pdf[label].values
    splits[label] = split_idx(y)
    n_tr, n_va, n_te = (len(splits[label][0]), len(splits[label][1]), len(splits[label][2]))
    print(f"{label}: train={n_tr}, val={n_va}, test={n_te}")


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 48, Finished, Available, Finished, False)

y_binary: train=80327, val=10041, test=10041
y_5class: train=80327, val=10041, test=10041


In [9]:
# ============================================================
# 6) METRIC HELPERS — EarlyRisk-MOOC metric set
# ============================================================
def evaluate_classifier(y_true, y_pred, y_proba=None, classes=None, name=""):
    res = {"name": name}
    res["accuracy"]    = accuracy_score(y_true, y_pred)
    res["macro_f1"]    = f1_score(y_true, y_pred, average="macro", zero_division=0)
    res["weighted_f1"] = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    if classes is None:
        classes = sorted(list(set(y_true) | set(y_pred)))
    per_class_f1 = f1_score(y_true, y_pred, average=None,
                            labels=classes, zero_division=0)
    for cls, f in zip(classes, per_class_f1):
        res[f"f1_{cls}"] = float(f)
    if y_proba is not None and len(classes) >= 2:
        try:
            if len(classes) == 2:
                res["auc"] = roc_auc_score(y_true, y_proba[:, 1])
            else:
                res["auc_ovr"] = roc_auc_score(
                    y_true, y_proba, multi_class="ovr",
                    labels=classes, average="macro")
        except Exception as e:
            res["auc_error"] = str(e)
    return res


def flatten_result(res, name, classes_order, runtime, W, label_col):
    out = {"name": name, "label": label_col, "window": W,
           "runtime_sec": round(runtime, 2)}
    for k, v in res.items():
        if k.startswith("f1_") and k != "f1_score":
            try:
                idx = int(k.split("_")[1])
                out[f"f1_{classes_order[idx]}"] = v
            except (ValueError, IndexError):
                out[k] = v
        elif k in ("accuracy","macro_f1","weighted_f1","auc","auc_ovr"):
            out[k] = v
    return out


print("Metric helpers ready")


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 49, Finished, Available, Finished, False)

Metric helpers ready


In [10]:
# ============================================================
# 7) IMPUTATION + SCALING (fit on TRAIN only)
# ============================================================
def fit_impute_scale(X_tr, X_te, X_va=None):
    imp = SimpleImputer(strategy="median").fit(X_tr)
    X_tr_i = imp.transform(X_tr)
    X_te_i = imp.transform(X_te)
    sc = StandardScaler().fit(X_tr_i)
    out = (sc.transform(X_tr_i), sc.transform(X_te_i))
    if X_va is not None:
        out = out + (sc.transform(imp.transform(X_va)),)
    return out

print("fit_impute_scale ready")


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 51, Finished, Available, Finished, False)

fit_impute_scale ready


In [11]:
# ============================================================
# 8) AOUARIB — GA + XGBoost
# ============================================================
XGB_AVAILABLE = False
try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception:
    from sklearn.ensemble import GradientBoostingClassifier


LR_GRID, N_EST_GRID, DEPTH_GRID = [0.05, 0.1, 0.2], [100, 200, 400], [3, 6, 9]
SUBS_GRID, COLS_GRID = [0.6, 0.8, 1.0], [0.6, 0.8, 1.0]
POP, GENS, ELITE, MUT = 12, 5, 2, 0.10


def make_xgb(p, n_cls):
    if XGB_AVAILABLE:
        return XGBClassifier(
            n_estimators=p["n_estimators"], max_depth=p["max_depth"],
            learning_rate=p["learning_rate"], subsample=p["subsample"],
            colsample_bytree=p["colsample_bytree"],
            objective="binary:logistic" if n_cls == 2 else "multi:softprob",
            eval_metric="logloss" if n_cls == 2 else "mlogloss",
            random_state=SEED, n_jobs=-1, verbosity=0, use_label_encoder=False)
    return GradientBoostingClassifier(
        n_estimators=p["n_estimators"], max_depth=p["max_depth"],
        learning_rate=p["learning_rate"], subsample=p["subsample"],
        random_state=SEED)


def random_chrom(n_feats):
    return {
        "mask": np.random.randint(0, 2, size=n_feats).astype(bool),
        "n_estimators": int(np.random.choice(N_EST_GRID)),
        "max_depth":    int(np.random.choice(DEPTH_GRID)),
        "learning_rate":  float(np.random.choice(LR_GRID)),
        "subsample":      float(np.random.choice(SUBS_GRID)),
        "colsample_bytree": float(np.random.choice(COLS_GRID)),
    }


def fitness(c, Xtr, ytr, Xva, yva, n_cls):
    if c["mask"].sum() < 2: return 0.0
    try:
        m = make_xgb(c, n_cls)
        m.fit(Xtr[:, c["mask"]], ytr)
        return f1_score(yva, m.predict(Xva[:, c["mask"]]),
                        average="macro", zero_division=0)
    except Exception:
        return 0.0


def crossover(p1, p2, n_feats):
    cut = np.random.randint(1, n_feats - 1)
    return {
        "mask": np.concatenate([p1["mask"][:cut], p2["mask"][cut:]]),
        "n_estimators":   np.random.choice([p1["n_estimators"], p2["n_estimators"]]),
        "max_depth":      np.random.choice([p1["max_depth"],    p2["max_depth"]]),
        "learning_rate":  np.random.choice([p1["learning_rate"],p2["learning_rate"]]),
        "subsample":      np.random.choice([p1["subsample"],    p2["subsample"]]),
        "colsample_bytree": np.random.choice([p1["colsample_bytree"], p2["colsample_bytree"]]),
    }


def mutate(c, n_feats):
    if np.random.rand() < MUT:
        flip = np.random.rand(n_feats) < MUT
        c["mask"] = np.where(flip, ~c["mask"], c["mask"])
    if np.random.rand() < MUT: c["n_estimators"]   = int(np.random.choice(N_EST_GRID))
    if np.random.rand() < MUT: c["max_depth"]      = int(np.random.choice(DEPTH_GRID))
    if np.random.rand() < MUT: c["learning_rate"]  = float(np.random.choice(LR_GRID))
    if np.random.rand() < MUT: c["subsample"]      = float(np.random.choice(SUBS_GRID))
    if np.random.rand() < MUT: c["colsample_bytree"] = float(np.random.choice(COLS_GRID))
    return c


def ga_search(Xtr, ytr, Xva, yva, n_cls):
    n_feats = Xtr.shape[1]
    pop = [random_chrom(n_feats) for _ in range(POP)]
    for g in range(GENS):
        scored = sorted(
            [(c, fitness(c, Xtr, ytr, Xva, yva, n_cls)) for c in pop],
            key=lambda t: t[1], reverse=True)
        print(f"   gen {g+1}/{GENS}: best={scored[0][1]:.4f}  feats={int(scored[0][0]['mask'].sum())}")
        elites = [c for c, _ in scored[:ELITE]]
        children = list(elites)
        while len(children) < POP:
            i, j = np.random.choice(POP // 2, size=2, replace=False)
            children.append(mutate(crossover(scored[i][0], scored[j][0], n_feats), n_feats))
        pop = children
    final = sorted(
        [(c, fitness(c, Xtr, ytr, Xva, yva, n_cls)) for c in pop],
        key=lambda t: t[1], reverse=True)
    return final[0][0]


all_results = []
for label_col, classes in [("y_binary", [0,1]), ("y_5class", LABEL_ORDER_5)]:
    idx_tr, idx_va, idx_te = splits[label_col]
    y_all = pdf[label_col].values
    n_cls = len(classes)
    for W in WINDOWS:
        X = window_views[W]["X"].values
        X_tr_s, X_te_s, X_va_s = fit_impute_scale(X[idx_tr], X[idx_te], X[idx_va])

        # Plain XGB
        plain_p = {"n_estimators": 300, "max_depth": 6, "learning_rate": 0.1,
                   "subsample": 0.8, "colsample_bytree": 0.8}
        plain = make_xgb(plain_p, n_cls)
        print(f"[{label_col}] W={W} PlainXGB ...")
        t0 = time.time(); plain.fit(X_tr_s, y_all[idx_tr]); rt = time.time()-t0
        y_pred = plain.predict(X_te_s)
        y_proba = plain.predict_proba(X_te_s) if hasattr(plain, "predict_proba") else None
        res = evaluate_classifier(y_all[idx_te], y_pred, y_proba,
                                   classes=list(range(n_cls)), name="PlainXGB")
        all_results.append(flatten_result(res, "PlainXGB", classes, rt, W, label_col))

        # GA-XGB
        print(f"[{label_col}] W={W} GA-XGB (pop={POP}, gens={GENS}) ...")
        t0 = time.time()
        best = ga_search(X_tr_s, y_all[idx_tr], X_va_s, y_all[idx_va], n_cls)
        final = make_xgb(best, n_cls)
        final.fit(X_tr_s[:, best["mask"]], y_all[idx_tr])
        rt = time.time() - t0
        y_pred = final.predict(X_te_s[:, best["mask"]])
        y_proba = final.predict_proba(X_te_s[:, best["mask"]]) if hasattr(final, "predict_proba") else None
        res = evaluate_classifier(y_all[idx_te], y_pred, y_proba,
                                   classes=list(range(n_cls)), name="GA-XGB")
        row = flatten_result(res, "GA-XGB", classes, rt, W, label_col)
        row["n_selected_feats"] = int(best["mask"].sum())
        all_results.append(row)

res_df = pd.DataFrame(all_results)
print("\n=== Aouarib et al. 2025 — V4 PlainXGB vs GA-XGB ===")
print(res_df.round(4).to_string(index=False))
delta = (res_df.pivot_table(index=["label","window"],
                             columns="name", values="macro_f1")
                .assign(ga_gain=lambda d: d.get("GA-XGB", 0) - d.get("PlainXGB", 0))
                .round(4))
print("\n=== Macro-F1 — GA gain ===")
print(delta.to_string())
res_df.to_csv("/tmp/results_aouarib_2025.csv", index=False)
print("\nSaved -> /tmp/results_aouarib_2025.csv")


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 53, Submitted, Running, Running, True)

[y_binary] W=1 PlainXGB ...


[y_binary] W=1 GA-XGB (pop=12, gens=5) ...


   gen 1/5: best=0.9133  feats=13


   gen 2/5: best=0.9172  feats=15


...

   gen 3/5: best=0.8623  feats=17


   gen 4/5: best=0.8671  feats=16


   gen 5/5: best=0.8671  feats=16


[y_5class] W=4 PlainXGB ...


[y_5class] W=4 GA-XGB (pop=12, gens=5) ...
